In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


def base_lstm_model_01(input_shape):
    model = Sequential(
        [
            LSTM(64, return_sequences=False, input_shape=input_shape),
            Dropout(0.2),
            Dense(1),
        ]
    )
    return model


def base_lstm_model_02(input_shape):
    model = Sequential(
        [
            LSTM(64, return_sequences=True, input_shape=input_shape),
            Dropout(0.3),
            LSTM(32),
            Dense(1),
        ]
    )
    return model

In [ ]:
import os
from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint


def get_callbacks(model_name):
    tensorboard_cb = TensorBoard(log_dir=f"logs/{model_name}")
    checkpoint_cb = ModelCheckpoint(
        f"saved_models/{model_name}.h5", save_best_only=True
    )
    return [tensorboard_cb, checkpoint_cb]

In [9]:
import mlflow.keras
from tensorflow.keras.callbacks import LambdaCallback


log_metrics_callback = LambdaCallback(
    on_epoch_end=lambda epoch, logs: [
        mlflow.log_metric("loss", logs["loss"], step=epoch),
        mlflow.log_metric("mae", logs["mae"], step=epoch),
        mlflow.log_metric("val_loss", logs["val_loss"], step=epoch),
        mlflow.log_metric("val_mae", logs["val_mae"], step=epoch),
    ]
)

In [ ]:
import mlflow
import mlflow.tensorflow
import numpy as np
from utils import *
from utils import get_callbacks
from tensorflow.keras.optimizers import Adam

# Dummy Data (replace with real preprocessing)
X = np.random.randn(500, 30, 1)
y = np.random.randn(500, 1)

input_shape = (30, 1)


def train_model(model_fn, model_name, lr=0.001, epochs=10):
    with mlflow.start_run(run_name=model_name):
        mlflow.tensorflow.autolog()

        model = model_fn(input_shape)
        model.compile(optimizer=Adam(lr), loss="mse", metrics=["mae"])

        # Define callbacks
        callbacks = get_callbacks(model_name) + [log_metrics_callback]

        history = model.fit(
            X, y, epochs=epochs, validation_split=0.2, callbacks=callbacks
        )

        # Logging parameters

        mlflow.log_param("model_name", model_name)
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("epochs", epochs)
        mlflow.keras.log_model(model, f"{model_name}_model")

        return model, history


# Run both experiments
model_1, _ = train_model(base_lstm_model_01, "LSTM_Model_01", lr=0.00025)
model_2, _ = train_model(base_lstm_model_02, "LSTM_Model_02", lr=0.0005)

Epoch 1/10
 8/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.0754 - mae: 0.8648

13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step - loss: 1.0591 - mae: 0.8450 - val_loss: 1.3339 - val_mae: 0.9199
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 1.0518 - mae: 0.8191 - val_loss: 1.3356 - val_mae: 0.9216
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.9780 - mae: 0.7901 - val_loss: 1.3356 - val_mae: 0.9220
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.9994 - mae: 0.8127 - val_loss: 1.3365 - val_mae: 0.9230
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.9212 - mae: 0.7790 - val_loss: 1.3394 - val_mae: 0.9253
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 1.0660 - mae: 0.8363 - val_loss: 1.3388 - val_mae: 0.9250
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 1.0544 - mae: 0.8253 - val_loss: 1.3409 - val_mae: 0.9265
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.9730 - mae: 0.7752 - val_loss: 1.3395 - val_mae: 0.9255
Epoch 9/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 1.0084 - mae: 0.

2025/07/17 09:15:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/07/17 09:15:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/07/17 09:15:45 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
2025/07/17 09:15:55 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Epoch 1/10
10/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 1.1364 - mae: 0.8699

13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 66ms/step - loss: 1.1024 - mae: 0.8514 - val_loss: 1.3297 - val_mae: 0.9182
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 1.0241 - mae: 0.8030 - val_loss: 1.3370 - val_mae: 0.9234
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 1.1181 - mae: 0.8369 - val_loss: 1.3442 - val_mae: 0.9283
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - loss: 1.0562 - mae: 0.8225 - val_loss: 1.3428 - val_mae: 0.9272
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - loss: 1.0223 - mae: 0.8138 - val_loss: 1.3452 - val_mae: 0.9289
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - loss: 1.0324 - mae: 0.8169 - val_loss: 1.3497 - val_mae: 0.9318
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - loss: 0.9843 - mae: 0.7967 - val_loss: 1.3461 - val_mae: 0.9296
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - loss: 1.0196 - mae: 0.8011 - val_loss: 1.3438 - val_mae: 0.9277
Epoch 9/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 1.0147 - mae: 0.

2025/07/17 09:16:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/07/17 09:16:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/07/17 09:16:13 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
2025/07/17 09:16:22 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
